#### **Problem Statement**

- Recent times have seen an increase in the use of large language models such as Gemini, ChatGPT, etc., for various applications.

- However, rapid deployment of these LLMs has raised concerns regarding their risks - including ethical issues, data biases, privacy and robustness. [1]

- AI Guardrails are safety mechanisms filtering the inputs or outputs of large language models (LLMs) confining the LLMs, *to provide a more secure range of function as well as compliance to ethical principles and mitigating hallucinations*. [1]

- Guardrails identify potential misuse in the query stage and try to prevent the LLM from providing a response which should not be given. [1]

- Examples of opensource guardrail systems are from GuardrailsAI, Llama Guard, NVIDIA's NeMO Guardrails.

- *Llama Guard developed by Meta, is based on the Llama2-7b architecture, focusing on human-AI conversation safety [1]* Llama guard classifies content in both LLM inputs (prompt classfication) and in LLM response (response classification). Generates text as part of the output, indicating whether a given prompt is safe as well potential category of safety violation. [3]

- *GuardrailsAI enables the user to add a structure, type and quality, defining the RAIL spec in XML or pydantic, defining "guard" and wrapping the LLM.*

- *NVIDIA NeMo Guardrail functions as an intermediary layer, enhancing the control and safety of LLM through the use of Colang.*

**References :**
1.  https://arxiv.org/pdf/2402.01822.pdf
2. https://attri.ai/blog
3. https://huggingface.co/meta-llama/LlamaGuard-7b

#### **Installing Dependencies**

In [ ]:
# Installing the required dependencies : guardrails-ai, typing, pydantic , pypdfium2, openai libraries
# Pinned versions (tested Oct 2026) so this notebook keeps working on Google Colab
!pip install -q openai==0.28.1 guardrails-ai==0.3.2 pydantic==2.4.2 tiktoken==0.5.2 rich pypdfium2
# !pip install openai==0.28 -q   # (replaced by the pinned install above)
# !pip install guardrails-ai -q   # (replaced by the pinned install above)
# !pip install rich -q   # (replaced by the pinned install above)
# !pip install typing -q   # (replaced by the pinned install above)
# !pip install pydantic -q   # (replaced by the pinned install above)
# !pip install pypdfium2 -q   # (replaced by the pinned install above)

#### **Importing Dependencies**

In [ ]:
# Importing dependencies : openai, guardrails, google drive within google colab
import openai
import guardrails as gd
from rich import print
from google.colab import drive

In [ ]:
# Importing BaseModel, Field from pydantic, models are classes which inherit from pydantic.BaseModel
# https://docs.pydantic.dev/latest/concepts/models/
# Importing List from typing module
# Field function in pydantic is useful to customize and add metadata to fields of models
from pydantic import BaseModel, Field
from typing import List

#### **OPENAI API KEY**

In [ ]:
# storing the OpenAI API Key within google colab

from google.colab import userdata
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
openai.api_key = OPENAI_API_KEY

In [ ]:
# Mounting google drive
content = drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# using read_pdf() function from guardrails to load the pdf as a string
data = gd.docs_utils.read_pdf("/content/drive/MyDrive/Gemma_Google.pdf")

#### **Define Pydantic Model : Using Guardrail with Chat Models**

In [ ]:
# Instructions are provided, useful to generate system message for the chat model
# What goes in prompt, is split across prompt and instructions shown below :
# https://www.guardrailsai.com/docs/examples/guardrails_with_chat_models

instructions = """

You are a helpful assistant, only able to provide a response in a JSON format
and no other text.

${gr.json_suffix_prompt_examples}"""

prompt = """ Given the following document, provide a response the following questions. If the answer is
not in the document, please say `Not in the document`.

${document}

Extract information from this document and return a JSON object following the
correct schema.

${gr.xml_prefix_prompt}

${output_schema}

"""

class Model(BaseModel):
  name: str = Field(
      description = "What is the name of the Model discussed in the article?"
  )
  model_variant: str = Field(
      description = "Two sizes of the model weights that have been released"
  )
  integrating_tools: str = Field(
      description = "What tools does the model integrate with?"
  )

class BulletPoints(BaseModel):
  points : List[Model] = Field(
      description = """What key aspects have been discussed in this article regarding
       model variants?"""
  )

In [ ]:
# Creating gd.Guard object for validating and correcting the output LLM
guard = gd.Guard.from_pydantic(
    output_class = BulletPoints,
    instructions = instructions,
    prompt = prompt,
)

In [ ]:
print(guard.base_prompt)

Given the following document, provide a response the following questions. If the answer is
not in the document, please say `Not in the document`.

${document}

Extract information from this document and return a JSON object following the
correct schema.


Given below is XML that describes the information to extract from this document and the tags to extract it into.


<output>
    <list name="points" description="What key aspects have been discussed in this article regarding&#10;       
model variants?">
        <object>
            <string name="name" description="What is the name of the Model discussed in the article?"/>
            <string name="model_variant" description="Two sizes of the model weights that have been released"/>
            <string name="integrating_tools" description="What tools does the model integrate with?"/>
        </object>
    </list>
</output>

In [ ]:
#  Instructions sent as the system message to the LLM
raw_llm_response, validated_response, *rest = guard(
    openai.ChatCompletion.create,
    model = "gpt-4o-mini",
    prompt_params = {"document": data},
    max_tokens = 2048,
    temperature = 0.2
)

In [ ]:
print(validated_response)  # display the validated response

{
    'points': [
        {
            'name': 'Gemma',
            'model_variant': 'Gemma 2B and Gemma 7B',
            'integrating_tools': 'Hugging Face, MaxText, NVIDIA NeMo and TensorRT-LLM'
        }
    ]
}

In [ ]:
#accessing logs via guard.history
guard.history.last.tree

Logs
├── ╭────────────────────────────────────────────────── Step 0 ───────────────────────────────────────────────────╮
│   │ ╭──────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────╮ │
│   │ │  Given the following document, provide a response the following questions. If the answer is             │ │
│   │ │ not in the document, please say `Not in the document`.                                                  │ │
│   │ │                                                                                                         │ │
│   │ │ 2/23/24, 12:17 PM Gemma: Google introduces new state-of-the-art open models                             │ │
│   │ │ https://blog.google/technology/developers/gemma-open-models/ 1/5                                        │ │
│   │ │ DEVELOPERS                                                                                              │ │
│   │ │ Gemma: Introducing new state-of-the-art                                                                 │ │
│   │ │ open models                                                                                             │ │
│   │ │ Feb 21, 2024 · 3 min read                                                                               │ │
│   │ │ Gemma is built for responsible AI development from the same research and technology used to             │ │
│   │ │ create Gemini models.                                                                                   │ │
│   │ │ J                                                                                                       │ │
│   │ │ Jeanine Banks                                                                                           │ │
│   │ │ VP & GM, Developer X and DevRel                                                                         │ │
│   │ │ T                                                                                                       │ │
│   │ │ Tris Warkentin                                                                                          │ │
│   │ │ Director, Google DeepMind                                                                               │ │
│   │ │ Listen to article                                                                                       │ │
│   │ │ 7 minutes                                                                                               │ │
│   │ │ The Keyword                                                                                             │ │
│   │ │ 2/23/24, 12:17 PM Gemma: Google introduces new state-of-the-art open models                             │ │
│   │ │ https://blog.google/technology/developers/gemma-open-models/ 2/5                                        │ │
│   │ │ At Google, we believe in making AI helpful for everyone. We have a long history of contributing         │ │
│   │ │ innovations to                                                                                          │ │
│   │ │ the open community, such as with Transformers, TensorFlow, BERT, T5, JAX, AlphaFold, and AlphaCode.     │ │
│   │ │ Today,                                                                                                  │ │
│   │ │ we’re excited to introduce a new generation of open models from Google to assist developers and         │ │
│   │ │ researchers                                                                                             │ │
│   │ │ in building AI responsibly.                                                                             │ │
│   │ │ Gemma open models                                                                                       │ │
│   │ │ Gemma is a family of lightweight, state-of-the-art open models built from the same research and         │ │
│   │ │ technology                                                                                              │ │
│   │ │ used to create the Gemini models. Developed